# Lab 5: Implementation of Ordinary Least Squares (OLS) and Gradient Descent for Linear Regression

**Course**: Machine Learning  
**Topic**: Implementation of Ordinary Least Squares (OLS) and Gradient Descent for Linear Regression  
**Instructor**: Sharad Laad | ORY AI Labs  

---

## 1. Objectives
- Implement simple linear regression from scratch in Python using two foundational paradigms:
  1. **Analytical Approach**: Ordinary Least Squares (OLS) closed-form solution.
  2. **Iterative Approach**: Batch Gradient Descent optimization.
- Train both implementations on a sample bivariate dataset ($X = \text{Height}$, $y = \text{Weight}$).
- Compare and verify the resulting parameters ($m$ and $c$).
- Compute statistical evaluation metrics: Coefficient of Determination ($R^2$), Standard Error ($SE(m)$), $t$-statistic, and two-tailed $p$-value for slope significance.
- Visualize the fitted regression lines against the ground truth data points.
- Analyze loss convergence behavior across iterations and sensitivity to learning rates.

## 2. Problem Statement & Dataset

Given the experimental dataset of Height ($X$) and Weight ($y$):

| Observation ($i$) | Height ($X$) | Weight ($y$) |
| :---: | :---: | :---: |
| 1 | 0.5 | 1.4 |
| 2 | 2.3 | 1.9 |
| 3 | 2.9 | 3.2 |

Find the best-fit line:
$$y = mx + c$$
where:
- $m$ is the slope (weight parameter)
- $c$ is the y-intercept (bias parameter)

## 3. Mathematical Foundations

### Method 1: Ordinary Least Squares (Closed-Form Analytical Solution)
The closed-form analytical solution minimizes the sum of squared residuals directly:
$$\bar{x} = \frac{1}{n} \sum_{i=1}^n x_i, \quad \bar{y} = \frac{1}{n} \sum_{i=1}^n y_i$$
$$m = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}$$
$$c = \bar{y} - m\bar{x}$$

### Method 2: Batch Gradient Descent (Iterative Optimization)
1. **Cost Function (Mean Squared Error - MSE)**:
   $$J(m, c) = \frac{1}{n}\sum_{i=1}^n (\hat{y}_i - y_i)^2 = \frac{1}{n}\sum_{i=1}^n (mx_i + c - y_i)^2$$
2. **Partial Derivatives (Gradients)**:
   $$\frac{\partial J}{\partial m} = \frac{2}{n} \sum_{i=1}^n (mx_i + c - y_i) \cdot x_i$$
   $$\frac{\partial J}{\partial c} = \frac{2}{n} \sum_{i=1}^n (mx_i + c - y_i)$$
3. **Parameter Update Rules** (with learning rate $\alpha$):
   $$m := m - \alpha \frac{\partial J}{\partial m}$$
   $$c := c - \alpha \frac{\partial J}{\partial c}$$

### Method 3: Statistical Evaluation Metrics ($R^2$, $t$-statistic, and $p$-value)
1. **Coefficient of Determination ($R^2$)**:
   $$SS_{res} = \sum_{i=1}^n (y_i - \hat{y}_i)^2, \quad SS_{tot} = \sum_{i=1}^n (y_i - \bar{y})^2$$
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$
2. **Hypothesis Testing on Slope $m$**:
   - $H_0: m = 0$ (No linear relationship between Height and Weight)
   - $H_1: m \neq 0$ (Statistically significant relationship)
   - Residual Variance: $s_e^2 = \frac{SS_{res}}{n - 2}$ (degrees of freedom $df = n - 2$)
   - Standard Error of Slope: $SE(m) = \sqrt{\frac{s_e^2}{\sum (x_i - \bar{x})^2}}$
   - $t$-Statistic: $t = \frac{m - 0}{SE(m)}$
   - Two-Tailed $p$-value: $p = 2 \cdot (1 - F(|t|, df)) = 2 \cdot \text{sf}(|t|, df)$

## 4. Implementation

### Task 1: Environment Setup and Dataset Definition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Dataset Definition
X = np.array([0.5, 2.3, 2.9])
y = np.array([1.4, 1.9, 3.2])

print(f"X (Height): {X}")
print(f"y (Weight): {y}")
print(f"Sample size n = {len(X)}, Degrees of freedom df = {len(X) - 2}")

### Task 2: Implement OLS Function (Analytical Approach)

In [ ]:
def ols_linear_regression(X, y):
    """
    Compute linear regression parameters analytically using OLS closed-form formulas.
    
    Parameters:
        X (np.ndarray): Input feature vector (Height)
        y (np.ndarray): Target vector (Weight)
        
    Returns:
        m (float): Optimal slope
        c (float): Optimal intercept
    """
    # Compute mean of X and mean of y
    x_mean = np.mean(X)
    y_mean = np.mean(y)
    
    # Compute numerator and denominator for slope 'm'
    numerator = np.sum((X - x_mean) * (y - y_mean))
    denominator = np.sum((X - x_mean) ** 2)
    m = numerator / denominator
    
    # Compute intercept 'c'
    c = y_mean - m * x_mean
    
    return float(m), float(c)

m_ols, c_ols = ols_linear_regression(X, y)
print(f"OLS Closed-Form Solution: m = {m_ols:.6f}, c = {c_ols:.6f}")

### Task 3: Implement Batch Gradient Descent (Iterative Approach)

In [ ]:
def gradient_descent(X, y, alpha=0.05, epochs=1000):
    """
    Optimize linear regression parameters using batch gradient descent.
    
    Parameters:
        X (np.ndarray): Input feature vector
        y (np.ndarray): Target vector
        alpha (float): Learning rate
        epochs (int): Total training iterations
        
    Returns:
        m (float): Learned slope
        c (float): Learned intercept
        loss_history (list): MSE loss recorded per epoch
    """
    m = 0.0
    c = 0.0
    n = len(X)
    loss_history = []
    
    for epoch in range(epochs):
        # Compute linear predictions
        y_pred = m * X + c
        
        # Compute Mean Squared Error (MSE) loss and store
        loss = np.mean((y_pred - y) ** 2)
        loss_history.append(float(loss))
        
        # Compute partial derivatives dm and dc
        dm = (2.0 / n) * np.sum((y_pred - y) * X)
        dc = (2.0 / n) * np.sum(y_pred - y)
        
        # Update parameters m and c using gradients and learning rate alpha
        m = m - alpha * dm
        c = c - alpha * dc
        
    return float(m), float(c), loss_history

m_gd, c_gd, losses = gradient_descent(X, y, alpha=0.05, epochs=1500)
print(f"Gradient Descent Solution (epochs=1500, alpha=0.05): m = {m_gd:.6f}, c = {c_gd:.6f}")

### Task 4: Implement Statistical Evaluation Metrics ($R^2$, $t$-statistic, $p$-value)

In [ ]:
def compute_evaluation_metrics(X, y, y_pred, m):
    """
    Calculate R-squared and statistical significance (p-value) of the slope parameter.
    """
    n = len(X)
    df = n - 2
    
    # SS_res and SS_tot
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    
    # R-squared
    r2 = 1.0 - (ss_res / ss_tot)
    
    # Standard error of the slope SE(m)
    residual_variance = ss_res / df if df > 0 else 0.0
    denom = np.sum((X - np.mean(X)) ** 2)
    se_m = np.sqrt(residual_variance / denom)
    
    # t-statistic
    t_stat = m / se_m if se_m > 0 else np.nan
    
    # Two-tailed p-value using Student's t distribution survival function
    p_val = 2.0 * stats.t.sf(np.abs(t_stat), df=df) if df > 0 else np.nan
    
    return float(r2), float(t_stat), float(p_val)

# Evaluate OLS
y_pred_ols = m_ols * X + c_ols
r2_ols, t_ols, p_ols = compute_evaluation_metrics(X, y, y_pred_ols, m_ols)

# Evaluate GD
y_pred_gd = m_gd * X + c_gd
r2_gd, t_gd, p_gd = compute_evaluation_metrics(X, y, y_pred_gd, m_gd)

print("==================== OLS RESULTS ====================")
print(f"Equation     : y = {m_ols:.4f}x + {c_ols:.4f}")
print(f"R-squared    : {r2_ols:.4f}")
print(f"t-statistic  : {t_ols:.4f}")
print(f"p-value      : {p_ols:.4f}\n")

print("============== GRADIENT DESCENT RESULTS =============")
print(f"Equation     : y = {m_gd:.4f}x + {c_gd:.4f}")
print(f"R-squared    : {r2_gd:.4f}")
print(f"t-statistic  : {t_gd:.4f}")
print(f"p-value      : {p_gd:.4f}\n")

### Task 5: Side-by-Side Comparison and Visualization

In [ ]:
print("================ SIDE-BY-SIDE COMPARISON ================")
print(f"{'Metric / Parameter':<22} | {'Analytical (OLS)':<16} | {'Iterative (GD)':<16} | {'Difference':<12}")
print("-" * 72)
print(f"{'Slope (m)':<22} | {m_ols:<16.6f} | {m_gd:<16.6f} | {abs(m_ols - m_gd):<12.6f}")
print(f"{'Intercept (c)':<22} | {c_ols:<16.6f} | {c_gd:<16.6f} | {abs(c_ols - c_gd):<12.6f}")
print(f"{'R-squared':<22} | {r2_ols:<16.6f} | {r2_gd:<16.6f} | {abs(r2_ols - r2_gd):<12.6f}")
print(f"{'t-statistic':<22} | {t_ols:<16.6f} | {t_gd:<16.6f} | {abs(t_ols - t_gd):<12.6f}")
print(f"{'p-value':<22} | {p_ols:<16.6f} | {p_gd:<16.6f} | {abs(p_ols - p_gd):<12.6f}")

# Create 2-panel plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Regression Fit Comparison
x_vals = np.linspace(0, 3.5, 100)
axes[0].scatter(X, y, color="#1f77b4", s=100, label="Actual Data Points", zorder=5)
axes[0].plot(x_vals, m_ols * x_vals + c_ols, color="#2ca02c", linestyle="-", linewidth=2.5, label=f"OLS: y = {m_ols:.2f}x + {c_ols:.2f}")
axes[0].plot(x_vals, m_gd * x_vals + c_gd, color="#d62728", linestyle="--", linewidth=2, label=f"GD: y = {m_gd:.2f}x + {c_gd:.2f}")
axes[0].set_title("OLS vs Gradient Descent Regression Line", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Height (X)", fontsize=11)
axes[0].set_ylabel("Weight (y)", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend(fontsize=10)

# Subplot 2: Loss Convergence Curve
axes[1].plot(range(1, len(losses) + 1), losses, color="#9467bd", linewidth=2)
axes[1].set_title("MSE Loss Convergence over Epochs", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Mean Squared Error (Loss)", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

### Task 6: Learning Rate Sensitivity Analysis

In [ ]:
rates = [1.0, 0.05, 0.0001]
plt.figure(figsize=(10, 5))

for lr in rates:
    m_tmp, c_tmp, l_tmp = gradient_descent(X, y, alpha=lr, epochs=100)
    print(f"alpha = {lr:<6} -> Final m={m_tmp:10.4f}, c={c_tmp:10.4f}, Final Loss={l_tmp[-1]:.4e}")
    display_losses = [min(loss, 100.0) for loss in l_tmp[:50]]
    plt.plot(range(1, len(display_losses) + 1), display_losses, label=f"alpha = {lr}")

plt.title("Learning Rate Sensitivity (First 50 Epochs)", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("MSE Loss (Capped at 100 for scale)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=10)
plt.show()

## 5. Lab Report & Viva Questions

### Question 1: Analytical vs. Iterative Comparison
**Do the values of $(m, c)$ and $R^2$ obtained via Gradient Descent match those from OLS? If there is a slight difference, why?**
- **Answer**: Yes, the values match to 6 decimal places ($m \approx 0.641026, c \approx 0.948718, R^2 \approx 0.7425$). Any slight numerical difference (on the order of $10^{-7}$ or less) arises because OLS calculates the exact global algebraic minimum in a single step using closed-form matrix calculus, whereas Batch Gradient Descent is an asymptotic iterative algorithm that takes finite step sizes $\alpha$ along the gradient, stopping after a finite number of epochs.

### Question 2: Interpreting $R^2$ vs. $p$-value for Small Sample Sizes ($n = 3$)
**Notice that the $R^2$ score is high ($0.7425$), indicating a strong correlation, yet the $p$-value for the slope ($0.3388$) exceeds the standard significance threshold ($\alpha = 0.05$). Explain why a high $R^2$ can coexist with a non-significant $p$-value when sample size $n$ is very small ($df = 1$).**
- **Answer**: $R^2$ measures the proportion of sample variance explained by the model, while the $p$-value tests whether this observed relationship is statistically distinguishable from random chance in the broader population. With $n = 3$, degrees of freedom is $df = n - 2 = 1$. The Student's $t$-distribution with 1 degree of freedom (equivalent to a Cauchy distribution) has extremely heavy, fat tails. The standard error of the slope $SE(m)$ is proportionally large ($0.3775$), yielding $t = 1.6981$. To achieve statistical significance at $\alpha = 0.05$ with $df = 1$, the critical $t$-value is $t_{0.025, 1} = 12.706$. Thus, even strong sample correlation cannot rule out chance when $n$ is tiny.

### Question 3: Hypothesis Testing on the Slope
**State the null hypothesis ($H_0$) and alternative hypothesis ($H_1$) tested by the $t$-statistic on the slope $m$.**
- **Answer**:
  - **Null Hypothesis ($H_0$)**: $\beta_1 = 0$ (There is no linear relationship between Height and Weight in the population; variations in Weight are independent of Height).
  - **Alternative Hypothesis ($H_1$)**: $\beta_1 \neq 0$ (There is a statistically significant linear relationship between Height and Weight).

### Question 4: Learning Rate Sensitivity
**What happens to parameter convergence if you increase the learning rate to $\alpha = 1.0$? What happens if you decrease the learning rate to $\alpha = 0.0001$?**
- **Answer**:
  - **When $\alpha = 1.0$**: The step size overshoots the minimum of the parabolic error surface. Consecutive gradient updates oscillate with increasing amplitude, causing the parameters to diverge toward infinity and the MSE loss to explode ($> 10^{198}$).
  - **When $\alpha = 0.0001$**: The updates are exceedingly conservative. After 100 epochs, the slope barely increases from $0.0$ to $0.0906$, and the model requires hundreds of thousands of epochs to approach the optimum.

### Question 5: Computational Scalability
**The closed-form OLS formulation requires computing $(X^T X)^{-1}$. Explain why Gradient Descent is preferred when training models on high-dimensional datasets with millions of features/samples.**
- **Answer**:
  - For a dataset with $n$ samples and $d$ features, computing $X^T X$ takes $\mathcal{O}(n d^2)$ time, and computing the matrix inverse $(X^T X)^{-1}$ requires $\mathcal{O}(d^3)$ time and $\mathcal{O}(d^2)$ memory storage.
  - When $d$ is very large (e.g., $10^5$ to $10^7$ features in text, genomics, or deep learning), matrix inversion becomes computationally intractable and exceeds system RAM.
  - In contrast, each iteration of Batch Gradient Descent (or Mini-batch/Stochastic Gradient Descent) requires only matrix-vector multiplications of order $\mathcal{O}(n d)$, requiring $\mathcal{O}(d)$ memory. This makes Gradient Descent vastly scalable for massive big-data settings.